# Comparing translated and independently generated neighbours

This example uses the repository's small synthetic vine, invented spacings and invented core positions in centimetres. Both arms use the same central vine, the same layout and the same observation volume. The example shows how to account for individual plants and interpret occupancy in one realization.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from deeprootgen.research.neighbourhood import generate_neighbourhood
from deeprootgen.research.core_observation import qualify_root_segments, predict_core_observations

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/synthetic_minimal.json").exists())
config = json.loads((root / "examples/synthetic_minimal.json").read_text())
origins = [(x, y) for x in (-8, 0, 8) for y in (-10, 0, 10)]
cores = [dict(core_id=f"{x}_{y}_{top}", x_cm=x, y_cm=y, z_upper_cm=-top, z_lower_cm=-top-5,
              radius_cm=2, sampled=True)
         for x in (-4, 0, 4) for y in (-5, 0, 5) for top in (0, 5, 10, 15)]

## Which vines reached the same cores?

The example sums the finite-cylinder contributions from all plants. The sum is checked for every depth increment, including increments with zero RLD. Focal and neighbour means below include zeros. The mean on a hit uses positive increments only.

In [2]:
rows = []
for mode in ("translated", "independent"):
    graph = generate_neighbourhood(config, origins, mode)
    focal = graph.plants.query("x_cm == 0 and y_cm == 0").plant_id.item()
    support = qualify_root_segments(graph.nodes, graph.edges, coordinate_frame="synthetic-soil-cm",
                                    fine_diameter_max_mm=2, dry_density_g_cm3=0.4)
    result = predict_core_observations(support, cores, by_plant=True)
    by_plant = result.by_plant.pivot(index="core_id", columns="plant_id", values="total_rld_cm_cm3")
    total = result.cores.set_index("core_id").total_rld_cm_cm3.reindex(by_plant.index)
    np.testing.assert_allclose(by_plant.sum(axis=1), total, atol=1e-12)
    rows.append(dict(mode=mode, seeds=graph.plants.seed.nunique(), empty_increments=(total == 0).sum(),
                     increments=len(total), focal_rld=by_plant[focal].mean(),
                     neighbour_rld=by_plant.drop(columns=focal).sum(axis=1).mean(),
                     positive_rld=total[total > 0].mean()))
print(pd.DataFrame(rows).round(5).to_string(index=False))

       mode  seeds  empty_increments  increments  focal_rld  neighbour_rld  positive_rld
 translated      1                22          36    0.00648        0.01360       0.05164
independent      9                23          36    0.00648        0.01245       0.05244


## Reviewing a field comparison locally

Read the paired comparison first. Check the change in empty increments and entire cores alongside RLD on a hit, and look at focal and neighbour contributions separately. A paired seed interval describes uncertainty across simulated vineyards at the same field locations.

Next read the balanced comparison. Its layouts use the same set of geometries equally often at every position. A remaining change in occupancy measures the effect of combining different vine shapes at fixed marginal root length. Results from all layouts within one block share the same vines.

For collaborators with the private notebook archive, the empirical tables, seed map and report are in the local `Calvin_private/site1_independent_neighbours/` folder beside this notebook. The preceding improvement brief is in `Calvin_private/site1_broad_search_050/IMPROVEMENT_OPTIONS.pdf`. Open these from the authorized local archive. The [placement experiment options](../../docs/research/PLACEMENT_EXPERIMENT_OPTIONS.md) summarize the proposed follow-ups.